# Tải mã nguồn và dọn hình ảnh

Phụ trách: Nguyễn Đình Tuấn — 24120237 (nhánh `feat/source-downloader`).

Chương này đánh giá module `src/crawler/source_downloader.py`: tải gói e-print LaTeX của arXiv, giải nén an toàn vào `data/raw/unzipped/{arxiv_id}/{version}/`, xóa ảnh/nhị phân rồi so sánh dung lượng **trước** và **sau** khi dọn.

Khoảng ID được giao: `2502.03235` → `2502.08234` (5.000 bài, tháng 02/2025).

**Nguồn số liệu** (đúng schema do `ResourceProfiler` và `storage.py` sinh ra, không thêm cột mới):

| Tệp | Trường được dùng |
| :--- | :--- |
| `data/shards/24120237/stats.json` | `status`, `is_mock`, `id_range`, `run_args.rate_limit_sec`, `phases.download_clean.{elapsed_sec,calls}`, `results.{total_ids,succeeded,failed,not_found,publications,source_size_before_bytes,source_size_after_bytes,failed_ids}`, `disk.dirs` |
| `data/shards/24120237/publications.parquet` | `id`, `source_status`, `size_before_bytes`, `size_after_bytes`, `versions[].{version,source_status,size_before_bytes,size_after_bytes}` |

**Quy ước:**

- `size_before_bytes` = tổng dung lượng **sau giải nén, trước khi xóa** (mọi tệp trong gói); `size_after_bytes` = dung lượng thư mục còn lại sau khi dọn. Cả hai **không** phải kích thước tệp `.tar.gz` tải về.
- 1 KB = 1024 byte, 1 MB = 1024² byte (cùng quy ước với `profiler.py`).
- Trạng thái nguồn hợp lệ: `success`, `partial_success` (cấp bài), `no_latex`, `pdf_only`, `error`. Bài không tồn tại trên arXiv (`not_found`) không có dòng trong `publications.parquet`; số này chỉ có trong `stats.json`.
- **Nguyên tắc:** chỉ kết luận từ số đo thật. Nếu chưa có `stats.json` thật, notebook dùng dữ liệu mẫu và in cảnh báo; các số đó **không** được đưa vào báo cáo cuối.

## Mục lục

1. Tải và phân loại mã nguồn
2. Giải nén và dọn hình ảnh
3. Dung lượng trước và sau
4. Kết quả và nhận xét

In [ ]:
import json
import shutil
import sys
import tarfile
import tempfile
import io
import gzip
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import Markdown, display

# Tìm thư mục gốc repo để notebook chạy được từ docs/ hoặc từ gốc.
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MEMBER_ID = "24120237"
ASSIGNED_RANGE = {"start_month": "2025-02", "start_id": 3235, "end_month": "2025-02", "end_id": 8234}
TOTAL_IDS = ASSIGNED_RANGE["end_id"] - ASSIGNED_RANGE["start_id"] + 1  # 5.000 bài
SHARD_DIR = ROOT / "data" / "shards" / MEMBER_ID

KB, MB = 1024, 1024**2
STATUS_ORDER = ["success", "partial_success", "no_latex", "pdf_only", "error"]
COLORS = {"before": "#8c9bb0", "after": "#2a7ab9", "success": "#2a7ab9", "partial_success": "#6fa8d6",
          "no_latex": "#e0a030", "pdf_only": "#b0b0b0", "error": "#c0392b", "missing": "#7f7f7f"}
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})


def fmt_bytes(n):
    if n is None or (isinstance(n, float) and np.isnan(n)):
        return "n/a"
    for unit, size in (("MB", MB), ("KB", KB)):
        if abs(n) >= size:
            return f"{n / size:,.2f} {unit}"
    return f"{n:,.0f} B"


def load_shard(shard_dir: Path):
    stats_path, pub_path = shard_dir / "stats.json", shard_dir / "publications.parquet"
    stats = json.loads(stats_path.read_text(encoding="utf-8")) if stats_path.exists() else None
    pubs = pq.read_table(pub_path).to_pandas() if pub_path.exists() else None
    return stats, pubs


stats, pubs = load_shard(SHARD_DIR)
DATA_MODE = "real"
if stats is None or pubs is None or stats.get("is_mock", True):
    # Chưa có số đo thật: dùng dữ liệu mẫu do generate_mock_data.py sinh, chỉ để kiểm tra notebook chạy được.
    from src.crawler.generate_mock_data import generate_all_mock_data

    logging.disable(logging.WARNING)
    MOCK_DIR = Path(tempfile.mkdtemp(prefix="mock_shards_"))
    generate_all_mock_data(str(MOCK_DIR))
    logging.disable(logging.NOTSET)
    stats, pubs = load_shard(MOCK_DIR / MEMBER_ID)
    DATA_MODE = "mock"

if DATA_MODE == "mock":
    display(Markdown(
        "> ⚠️ **DỮ LIỆU MẪU (`is_mock = true`).** Chưa tìm thấy `stats.json`/`publications.parquet` thật trong "
        f"`data/shards/{MEMBER_ID}/`. Mọi bảng và biểu đồ bên dưới chỉ để kiểm tra notebook, "
        "**không được dùng làm kết luận**. Sau khi chạy lệnh thu thập ở `tasks/TASK_24120237_NguyenDinhTuan.md`, "
        "chạy lại notebook để có số liệu thật."))
else:
    display(Markdown(f"> ✅ **Dữ liệu thật** từ `data/shards/{MEMBER_ID}/` (trạng thái lần chạy: `{stats.get('status')}`)."))
    if stats.get("status") != "completed":
        display(Markdown(f"> ⚠️ Lần chạy chưa hoàn tất (`{stats.get('status')}`: {stats.get('error')}). Số liệu có thể thiếu."))
    if stats.get("id_range") and {k: stats["id_range"].get(k) for k in ASSIGNED_RANGE} != ASSIGNED_RANGE:
        display(Markdown(f"> ⚠️ `id_range` trong stats.json ({stats['id_range']}) khác khoảng ID được giao."))

res = stats.get("results", {})
print(f"Số bài trong publications.parquet : {len(pubs):,}")
print(f"Cột                                : {list(pubs.columns)}")
print(f"results trong stats.json           : { {k: v for k, v in res.items() if k != 'failed_ids'} }")

## 1. Tải và phân loại mã nguồn

`download_and_clean_source` tải `https://export.arxiv.org/src/{id}{version}` theo luồng (chunk 64 KB, nghỉ ≥ 3,0 s giữa hai request, retry có giới hạn) rồi phân loại gói tải về:

| Trạng thái | Ý nghĩa |
| :--- | :--- |
| `success` | Có ít nhất một tệp `.tex` sau khi giải nén và dọn |
| `no_latex` | Không có `.tex` nhưng còn `.bib` |
| `pdf_only` | Tác giả chỉ nộp PDF (hoặc gói chỉ chứa ảnh/PDF), không có LaTeX |
| `error` | Lỗi mạng/HTTP, gói hỏng hoặc bị từ chối do không an toàn (đường dẫn thoát thư mục, quá lớn) |
| `not_found` | arXiv trả HTTP 404 — ghi nhận vào `results.not_found`, không có dòng trong Parquet |

Ở cấp bài, `storage.resolve_paper_source_status` gộp trạng thái các phiên bản: tất cả `success` → `success`; có `success` lẫn phiên bản khác → `partial_success`; ngược lại ưu tiên `error` > `no_latex` > `pdf_only`.

**Tỷ lệ tải thành công** được báo cáo ở ba mẫu số khác nhau để không gây hiểu nhầm:

- *Trên bài có trong Parquet*: `success / publications`.
- *Có LaTeX*: `(success + partial_success) / publications`.
- *Trên toàn khoảng được giao*: `success / 5.000`, tính cả bài `not_found` và bài lỗi chưa vào Parquet.

In [ ]:
paper_counts = pubs["source_status"].fillna("missing").value_counts()
paper_counts = paper_counts.reindex(STATUS_ORDER + ["missing"], fill_value=0)
n_pubs = int(paper_counts.sum())

status_df = pd.DataFrame({"số bài": paper_counts, "tỷ lệ (%)": paper_counts / n_pubs * 100})
status_df.index.name = "source_status (cấp bài)"
display(status_df.round(2))

n_success = int(paper_counts["success"])
n_latex = n_success + int(paper_counts["partial_success"])
n_not_found = int(res.get("not_found", 0) or 0)
n_failed = int(res.get("failed", 0) or 0)
assigned = res.get("total_ids") or TOTAL_IDS

rates = pd.Series({
    "success / bài trong Parquet": n_success / n_pubs * 100,
    "(success + partial_success) / bài trong Parquet": n_latex / n_pubs * 100,
    f"success / {assigned:,} ID được giao": n_success / assigned * 100,
}, name="tỷ lệ (%)")
display(rates.round(2).to_frame())

cover = pd.DataFrame({"số ID": {
    "Được giao": assigned,
    "Có trong publications.parquet": n_pubs,
    "Không tồn tại (not_found)": n_not_found,
    "Lỗi, chưa ghi vào Parquet (failed)": n_failed,
}})
display(cover)
if assigned != n_pubs + n_not_found + n_failed:
    print(f"Lưu ý: {assigned} ≠ {n_pubs} + {n_not_found} + {n_failed}; chênh lệch = {assigned - n_pubs - n_not_found - n_failed} "
          "(ví dụ lần chạy dùng --max-papers hoặc chưa hoàn tất).")
if res.get("failed_ids"):
    print("Một số ID lỗi:", dict(list(res["failed_ids"].items())[:5]))

In [ ]:
# Cấp phiên bản: mỗi dòng của cột `versions` (struct: version, created, size_before_bytes, size_after_bytes, source_status).
ver_rows = []
for pid, vs in zip(pubs["id"], pubs["versions"]):
    for v in (list(vs) if vs is not None else []):
        ver_rows.append({"id": pid, **v})
ver_df = pd.DataFrame(ver_rows)

ver_counts = ver_df["source_status"].fillna("missing").value_counts().reindex(STATUS_ORDER + ["missing"], fill_value=0)
per_paper_versions = ver_df.groupby("id").size()
print(f"Tổng số phiên bản: {len(ver_df):,} (trung bình {per_paper_versions.mean():.2f} phiên bản/bài, lớn nhất {per_paper_versions.max()})")
print(f"Phiên bản chưa có source_status riêng (None): {int(ver_counts['missing']):,} "
      "— xảy ra khi thượng nguồn chỉ có kết quả cấp bài.")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, counts, title in ((axes[0], paper_counts, "Cấp bài"), (axes[1], ver_counts, "Cấp phiên bản")):
    counts = counts[counts > 0]
    bars = ax.barh(counts.index[::-1], counts.values[::-1], color=[COLORS[k] for k in counts.index[::-1]])
    ax.bar_label(bars, labels=[f"{v:,} ({v / counts.sum() * 100:.1f}%)" for v in counts.values[::-1]], padding=3, fontsize=8)
    ax.set_xlim(0, counts.max() * 1.3)
    ax.set_xlabel("Số lượng")
    ax.set_title(f"Trạng thái tải nguồn — {title}")
plt.tight_layout()
plt.show()

In [ ]:
# Thời gian pha download_clean và cận dưới do rate limit (≥ 3,0 s/request, mỗi phiên bản là một request).
dc = stats.get("phases", {}).get("download_clean", {})
rate = stats.get("run_args", {}).get("rate_limit_sec", 3.0)
calls, elapsed = dc.get("calls"), dc.get("elapsed_sec")
if calls and elapsed:
    lower_bound = calls * rate
    display(pd.DataFrame({"giá trị": {
        "Số lần gọi download_and_clean_source": calls,
        "Tổng thời gian pha download_clean (giờ)": round(elapsed / 3600, 3),
        "Trung bình (giây/lần gọi)": round(elapsed / calls, 2),
        f"Cận dưới do rate limit {rate:g} s (giờ)": round(lower_bound / 3600, 3),
        "Thời gian thực / cận dưới": round(elapsed / lower_bound, 2),
    }}))
    if DATA_MODE == "mock":
        print("(Số liệu thời gian của dữ liệu mẫu là giá trị giả định, không có ý nghĩa đo lường.)")
else:
    print("stats.json chưa có phases.download_clean.")

## 2. Giải nén và dọn hình ảnh

Phần này kiểm tra **hành vi thật** của `download_and_clean_source` trên các gói tự dựng, không cần mạng: hàm tải mạng `_download_stream` được thay bằng hàm trả về byte cục bộ, còn toàn bộ phần phân loại, giải nén an toàn, dọn tệp, đo dung lượng và di chuyển thư mục vẫn là mã thật. Mỗi ca nêu rõ trạng thái kỳ vọng; bảng kết quả là giá trị đo được, không phải giá trị điền tay.

Các ca gồm: gói `.tar.gz` nhiều tệp (có ảnh, PDF, EPS), gói chỉ có `.bib`, gói chỉ có ảnh/PDF, tệp `.gz` đơn chứa một `.tex`, bài chỉ nộp PDF, HTTP 404, lỗi mạng, và gói độc hại có đường dẫn `../` (phải bị từ chối).

In [ ]:
import src.crawler.source_downloader as sd


def make_tar(members: dict[str, bytes], extra=None) -> bytes:
    buf = io.BytesIO()
    with tarfile.open(fileobj=buf, mode="w:gz") as tar:
        for name, data in members.items():
            info = tarfile.TarInfo(name)
            info.size = len(data)
            tar.addfile(info, io.BytesIO(data))
        if extra:
            extra(tar)
    return buf.getvalue()


rng = np.random.default_rng(0)
blob = lambda n: rng.integers(0, 256, n, dtype=np.uint8).tobytes()  # dữ liệu nhị phân khó nén như ảnh
TEX = rb"\documentclass{article}\begin{document}Hello arXiv\end{document}" * 20
BIB = b"@article{a, title={T}, author={A}, year={2025}}\n" * 10

CASES = {
    # id giả -> (mô tả, trạng thái kỳ vọng, nội dung do "mạng" trả về; None=404, "ERR"=lỗi mạng)
    "2502.90001": ("tar.gz: tex + bib + ảnh/PDF/EPS", "success",
                   make_tar({"main.tex": TEX, "refs.bib": BIB, "fig/a.png": blob(80_000), "fig/b.pdf": blob(60_000),
                             "fig/c.eps": blob(40_000), "fig/d.jpg": blob(30_000)})),
    "2502.90002": ("tar.gz: chỉ .bib", "no_latex", make_tar({"refs.bib": BIB, "fig.png": blob(20_000)})),
    "2502.90003": ("tar.gz: chỉ ảnh/PDF", "pdf_only", make_tar({"a.png": blob(10_000), "b.pdf": blob(10_000)})),
    "2502.90004": ("gz đơn: một .tex", "success", gzip.compress(TEX)),
    "2502.90005": ("chỉ nộp PDF", "pdf_only", b"%PDF-1.5\n" + blob(50_000)),
    "2502.90006": ("HTTP 404", "not_found", None),
    "2502.90007": ("lỗi mạng sau retry", "error", "ERR"),
    "2502.90008": ("tar.gz độc hại: ../evil.tex", "error", make_tar({"main.tex": TEX, "../evil.tex": b"x"})),
}


def fake_download_stream(url, dest_path, session, **kwargs):
    # Thay thế phần mạng; trả về cùng mã trạng thái nội bộ như _download_stream thật.
    versioned = url.rsplit("/", 1)[-1]
    payload = CASES[versioned.removesuffix("v1")][2]
    if payload is None:
        return 404
    if payload == "ERR":
        return 0
    if payload.startswith(b"%PDF-"):
        return sd._STATUS_EARLY_PDF
    Path(dest_path).write_bytes(payload)
    return 200


workdir = Path(tempfile.mkdtemp(prefix="sd_demo_"))
real_stream, real_rl = sd._download_stream, sd.rate_limiter.wait
sd._download_stream, sd.rate_limiter.wait = fake_download_stream, lambda *a, **k: None
try:
    rows = []
    for aid, (desc, expected, payload) in CASES.items():
        out = sd.download_and_clean_source(aid, str(workdir / "sources"), str(workdir / "unzipped"),
                                           rate_limit_sec=0.0, remove_archive=True)
        src = Path(out["source_dir"])
        kept = sorted(p.relative_to(src).as_posix() for p in src.rglob("*") if p.is_file()) if src.exists() else []
        measured = sd.get_dir_size_bytes(src) if src.exists() else 0
        rows.append({
            "arxiv_id": aid, "ca kiểm thử": desc, "kỳ vọng": expected, "trạng thái": out["status"],
            "size_before (B)": out["size_before_bytes"], "size_after (B)": out["size_after_bytes"],
            "tệp còn lại": ", ".join(kept) or "—",
            "đúng kỳ vọng": out["status"] == expected,
            "after ≤ before": out["size_after_bytes"] <= out["size_before_bytes"],
            "after = đo lại thư mục": out["size_after_bytes"] == measured,
        })
    leftover_archives = [p.name for p in (workdir / "sources").glob("*") if p.is_file()]
    escaped = (workdir / "unzipped" / "evil.tex").exists() or (workdir / "evil.tex").exists()
finally:
    sd._download_stream, sd.rate_limiter.wait = real_stream, real_rl

demo_df = pd.DataFrame(rows).set_index("arxiv_id")
display(demo_df)
print("Tất cả ca đúng kỳ vọng      :", bool(demo_df["đúng kỳ vọng"].all()))
print("size_after ≤ size_before     :", bool(demo_df["after ≤ before"].all()))
print("size_after khớp đo lại       :", bool(demo_df["after = đo lại thư mục"].all()))
print("Gói nén còn sót (remove_archive=True):", leftover_archives or "không")
print("Tệp thoát khỏi thư mục đích  :", "CÓ" if escaped else "không")
shutil.rmtree(workdir, ignore_errors=True)

In [ ]:
# Phép đo trước/sau trên ca nhiều tệp: dung lượng theo loại tệp trong gói gốc.
members = {"main.tex": TEX, "refs.bib": BIB, "refs.bbl": BIB, "style.sty": TEX[:200], "fig/a.png": blob(80_000),
           "fig/b.pdf": blob(60_000), "fig/c.eps": blob(40_000), "fig/d.jpg": blob(30_000)}
by_ext = pd.Series({n: len(d) for n, d in members.items()}).rename("byte")
by_ext.index = [Path(n).suffix for n in by_ext.index]
by_ext = by_ext.groupby(level=0).sum().sort_values(ascending=False)

ext_df = pd.DataFrame({"dung lượng (KB)": by_ext / KB,
                       "giữ lại sau khi dọn": [e in sd.ALLOWLIST_EXTENSIONS for e in by_ext.index]})
display(ext_df.round(2))

print("Allowlist khi giải nén (ALLOWLIST_EXTENSIONS):", sorted(sd.ALLOWLIST_EXTENSIONS))
print("Danh sách giữ theo đặc tả nhiệm vụ (KEEP_EXTENSIONS):", sorted(sd.KEEP_EXTENSIONS))
dropped_text = sorted(sd.KEEP_EXTENSIONS - sd.ALLOWLIST_EXTENSIONS)
print("Loại văn bản đặc tả muốn giữ nhưng hiện bị bỏ khi giải nén:", dropped_text)

## 3. Dung lượng trước và sau

So sánh `size_before_bytes` (toàn bộ nội dung đã giải nén) với `size_after_bytes` (sau khi xóa ảnh/nhị phân). Chỉ tính các bài **có LaTeX** (`success`, `partial_success`) và `size_before_bytes > 0`, vì bài `pdf_only`/`error`/`not_found` không có nội dung để so sánh. Dung lượng ở cấp bài là **tổng các phiên bản**.

*Mức giảm gộp* = `1 − Σ after / Σ before` (bị chi phối bởi các bài lớn); *mức giảm trung vị theo bài* = trung vị của `1 − after/before` từng bài (đại diện cho bài điển hình).

In [ ]:
latex_df = pubs[pubs["source_status"].isin(["success", "partial_success"]) & (pubs["size_before_bytes"] > 0)].copy()
latex_df["reduction"] = 1 - latex_df["size_after_bytes"] / latex_df["size_before_bytes"]

assert (latex_df["size_after_bytes"] <= latex_df["size_before_bytes"]).all(), "Có bài size_after > size_before"

n = len(latex_df)
sb, sa = latex_df["size_before_bytes"], latex_df["size_after_bytes"]
size_table = pd.DataFrame({
    "Trước khi xóa ảnh": [sb.mean(), sb.median(), sb.sum()],
    "Sau khi xóa ảnh": [sa.mean(), sa.median(), sa.sum()],
}, index=["Trung bình / bài", "Trung vị / bài", "Tổng"])
size_table["Giảm (%)"] = (1 - size_table["Sau khi xóa ảnh"] / size_table["Trước khi xóa ảnh"]) * 100
display(size_table.assign(
    **{c: size_table[c].map(fmt_bytes) for c in ["Trước khi xóa ảnh", "Sau khi xóa ảnh"]}
).round({"Giảm (%)": 2}))

agg_red = (1 - sa.sum() / sb.sum()) * 100
med_red = latex_df["reduction"].median() * 100
print(f"Số bài có LaTeX dùng để so sánh : {n:,}")
print(f"Mức giảm gộp (Σ)                 : {agg_red:.2f}%")
print(f"Mức giảm trung vị theo bài        : {med_red:.2f}%")
print(f"Bài không giảm (after = before)   : {(latex_df['reduction'] == 0).sum():,} ({(latex_df['reduction'] == 0).mean() * 100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
vals = [sb.mean() / KB, sa.mean() / KB]
bars = ax.bar(["Trước khi xóa ảnh", "Sau khi xóa ảnh"], vals, color=[COLORS["before"], COLORS["after"]], width=0.55)
ax.bar_label(bars, labels=[f"{v:,.1f} KB" for v in vals], padding=3)
ax.set_ylabel("Dung lượng trung bình mỗi bài (KB)")
ax.set_title(f"Dung lượng trung bình trước/sau (n = {n:,} bài)")
ax.set_ylim(0, max(vals) * 1.18)

ax = axes[1]
ax.hist(latex_df["reduction"] * 100, bins=np.linspace(0, 100, 21), color=COLORS["after"], edgecolor="white")
ax.axvline(med_red, color=COLORS["error"], ls="--", label=f"Trung vị {med_red:.1f}%")
ax.set_xlabel("Mức giảm dung lượng của mỗi bài (%)")
ax.set_ylabel("Số bài")
ax.set_title("Phân bố mức giảm theo bài")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Phân bố dung lượng (thang log, đơn vị KB) và đối chiếu với tổng trong stats.json.
fig, ax = plt.subplots(figsize=(5.5, 4))
pos = latex_df[latex_df["size_after_bytes"] > 0]
data = [sb[sb > 0] / KB, pos["size_after_bytes"] / KB]
bp = ax.boxplot(data, tick_labels=["Trước", "Sau"], patch_artist=True, showfliers=True,
                flierprops={"markersize": 3, "alpha": 0.4})
for patch, c in zip(bp["boxes"], [COLORS["before"], COLORS["after"]]):
    patch.set_facecolor(c)
ax.set_yscale("log")
ax.set_ylabel("Dung lượng mỗi bài (KB, thang log)")
ax.set_title("Phân bố dung lượng mã nguồn trước/sau khi xóa ảnh")
plt.tight_layout()
plt.show()

tot_before, tot_after = res.get("source_size_before_bytes"), res.get("source_size_after_bytes")
pub_before, pub_after = int(pubs["size_before_bytes"].fillna(0).sum()), int(pubs["size_after_bytes"].fillna(0).sum())
check = pd.DataFrame({
    "stats.json (results)": [tot_before, tot_after],
    "Σ publications.parquet": [pub_before, pub_after],
}, index=["size_before_bytes", "size_after_bytes"])
check["khớp"] = check["stats.json (results)"] == check["Σ publications.parquet"]
display(check)
if not check["khớp"].all():
    print("Lưu ý: stats.json cộng dồn theo lần chạy/checkpoint còn Parquet là bản gộp cuối; "
          "chênh lệch thường do bài lỗi/chạy lại. Cần kiểm tra nếu lệch lớn.")

In [ ]:
# Dung lượng trước/sau theo trạng thái (mọi bài, không chỉ bài có LaTeX) để thấy phần nào của dữ liệu bị loại.
by_status = (pubs.assign(source_status=pubs["source_status"].fillna("missing"))
             .groupby("source_status")[["size_before_bytes", "size_after_bytes"]]
             .agg(["count", "sum"]))
by_status.columns = ["số bài", "before (tổng, B)", "_c", "after (tổng, B)"]
by_status = by_status.drop(columns="_c")
by_status["before"] = by_status["before (tổng, B)"].map(fmt_bytes)
by_status["after"] = by_status["after (tổng, B)"].map(fmt_bytes)
display(by_status[["số bài", "before", "after"]].reindex([s for s in STATUS_ORDER + ["missing"] if s in by_status.index]))

# Dung lượng thư mục giải nén còn lại trên đĩa (nếu profiler có ghi).
dirs = stats.get("disk", {}).get("dirs", {})
if "raw_unzipped" in dirs:
    d = dirs["raw_unzipped"]
    print(f"data/raw/unzipped: trước chạy {fmt_bytes(d['before_bytes'])} → sau chạy {fmt_bytes(d['after_bytes'])}")
if "raw_sources" in dirs:
    d = dirs["raw_sources"]
    print(f"data/raw/sources : trước chạy {fmt_bytes(d['before_bytes'])} → sau chạy {fmt_bytes(d['after_bytes'])} "
          "(≈ 0 nếu remove_archive=True đã xóa hết gói nén tạm)")

## 4. Kết quả và nhận xét

Các số trong bảng dưới được **tính trực tiếp** từ `stats.json` và `publications.parquet`, không điền tay.

In [ ]:
summary = pd.DataFrame({"giá trị": {
    "Chế độ dữ liệu": "DỮ LIỆU MẪU — không dùng làm kết luận" if DATA_MODE == "mock" else "Dữ liệu thật",
    "ID được giao": f"{assigned:,}",
    "Bài có trong Parquet": f"{n_pubs:,}",
    "Không tồn tại (not_found)": f"{n_not_found:,}",
    "Lỗi chưa vào Parquet (failed)": f"{n_failed:,}",
    "Tỷ lệ success / bài trong Parquet": f"{n_success / n_pubs * 100:.2f}%",
    "Tỷ lệ có LaTeX (success + partial_success)": f"{n_latex / n_pubs * 100:.2f}%",
    "Tỷ lệ pdf_only": f"{paper_counts['pdf_only'] / n_pubs * 100:.2f}%",
    "Tỷ lệ no_latex": f"{paper_counts['no_latex'] / n_pubs * 100:.2f}%",
    "Tỷ lệ error": f"{paper_counts['error'] / n_pubs * 100:.2f}%",
    "Dung lượng TB trước dọn (bài có LaTeX)": fmt_bytes(sb.mean()),
    "Dung lượng TB sau dọn (bài có LaTeX)": fmt_bytes(sa.mean()),
    "Mức giảm gộp": f"{agg_red:.2f}%",
    "Mức giảm trung vị theo bài": f"{med_red:.2f}%",
    "Tổng trước → sau": f"{fmt_bytes(sb.sum())} → {fmt_bytes(sa.sum())}",
}})
display(summary)

invariants = {
    "size_after ≤ size_before ở mọi bài có LaTeX": bool((sa <= sb).all()),
    "Không bài nào error/pdf_only có size_after > 0 bất thường": bool(
        (pubs.loc[pubs["source_status"].isin(["error", "pdf_only"]), "size_after_bytes"].fillna(0) <= 0).all()),
    "Hành vi downloader đúng ở 8 ca kiểm thử offline (mục 2)": bool(demo_df["đúng kỳ vọng"].all()),
}
display(pd.DataFrame({"đạt": invariants}))

**Nhận xét** *(đọc cùng các bảng ở trên; khi chạy lại với dữ liệu thật hãy đối chiếu từng ý với số liệu mới)*:

- **Tải nguồn:** tỷ lệ `success` và tỷ lệ có LaTeX cho biết mức độ bài báo tháng 02/2025 còn phát hành mã nguồn. Phần `pdf_only` là giới hạn của dữ liệu (tác giả không nộp LaTeX), không phải lỗi của downloader; phần `error` mới phản ánh sự cố mạng/gói hỏng và có thể giảm bằng `--resume`.
- **Thời gian:** pha `download_clean` bị chi phối bởi rate limit 3,0 s/request chứ không phải băng thông hay CPU; mỗi bài có `N` phiên bản tốn ít nhất `N × 3,0` giây, nên tổng thời gian có cận dưới đã tính ở mục 1.
- **Dung lượng:** xóa ảnh/nhị phân loại phần lớn dung lượng vì ảnh, PDF, EPS nặng hơn văn bản nhiều lần; mức giảm trung vị theo bài và mức giảm gộp khác nhau do một số ít bài rất nặng (nhiều ảnh) chi phối tổng.
- **Giới hạn đã biết:** (1) allowlist giải nén hiện chỉ giữ `.tex/.bib`, bỏ `.bbl/.sty/.cls/.bst/.txt` dù đặc tả và nhánh dự phòng của `reference_scraper.py` có dùng `.bbl` — cần thống nhất với người tích hợp trước khi chạy toàn khoảng; (2) `size_before_bytes` là dung lượng **đã giải nén** nên không so sánh trực tiếp với kích thước tệp `.tar.gz`; (3) bài `pdf_only` và `not_found` không có dung lượng nên bị loại khỏi phép so sánh trước/sau.